# 95. The Supply Chain Network Design under Uncertainty Problem
## Tier 3: Advanced Algorithm (Genetic Algorithm for Multi-Objective Optimization)

### Key assumptions
- Multi-objective optimization balancing cost, risk, and service level
- Population-based search with genetic operators
- Chromosome encoding for facility locations and customer assignments
- Fitness evaluation across multiple uncertainty scenarios
- Evolutionary search with selection, crossover, and mutation

### Approach (step-by-step)
1. **Chromosome Design**: Encode facility locations and customer assignments
2. **Population Initialization**: Generate diverse initial solutions
3. **Fitness Evaluation**: Multi-objective fitness across scenarios
4. **Selection**: Tournament selection based on fitness
5. **Crossover**: Exchange genetic material between parents
6. **Mutation**: Introduce random variations for diversity
7. **Evolution**: Iterative improvement over generations
8. **Convergence**: Track progress and identify best solutions

### What to look for in the results
- Convergence behavior over generations
- Pareto frontier of cost-risk-service trade-offs
- Solution diversity and exploration capability
- Performance improvement vs heuristic methods

### Concrete example (from the source)
Complex network with 6 facilities, 10 customers, 4 uncertainty scenarios:
- Fixed costs: [200, 180, 220, 190, 210, 195]
- Demand scenarios: High, Medium-High, Medium-Low, Low
- Capacities: [180, 200, 160, 190, 210, 175]
- Scenario probabilities: [0.2, 0.3, 0.3, 0.2]
- Population: 100 chromosomes, 500 generations

### Visualization(s)
- Genetic algorithm convergence plots
- Pareto frontier visualization
- Diversity metrics over time
- Solution quality evolution

### Why this Tier exists vs Tiers 1-2
Tier 3 addresses complex multi-objective optimization where traditional methods struggle, providing global search capability and Pareto-optimal solutions.

### Pros / Cons vs Tiers 1-2
**Pros:**
- Handles multiple conflicting objectives simultaneously
- Global search capability avoiding local optima
- Generates diverse solution alternatives
- Adaptable to various problem structures

**Cons:**
- Computationally intensive for large populations
- Parameter tuning required (mutation rate, crossover probability)
- No convergence guarantee to global optimum
- Complex implementation compared to heuristics

### When to use this Tier
- Multi-objective optimization problems
- When solution diversity is valuable
- Complex search spaces with many local optima
- When Pareto-optimal alternatives are needed

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class GeneticInstance:
    """Data structure for genetic algorithm supply chain network design"""
    facilities: List[int]
    customers: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    demands: Dict[Tuple[int, int, int], float]  # (customer, product, scenario) -> demand
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int], float]  # (facility, customer) -> unit cost
    scenario_probs: List[float]

def create_genetic_instance():
    """Create a complex instance for genetic algorithm demonstration"""
    facilities = [0, 1, 2, 3, 4, 5]  # 6 facilities
    customers = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # 10 customers
    scenarios = [0, 1, 2, 3]  # 4 scenarios
    
    # Fixed costs for facilities
    fixed_costs = [200, 180, 220, 190, 210, 195]
    
    # Capacities
    capacities = [180, 200, 160, 190, 210, 175]
    
    # Scenario probabilities
    scenario_probs = [0.2, 0.3, 0.3, 0.2]
    
    # Transportation costs (facility, customer) - realistic cost matrix
    transport_costs = {}
    for i in facilities:
        for j in customers:
            # Base cost with distance-like variation
            base_cost = 2.0 + abs(i - j) * 0.3 + np.random.normal(0, 0.2)
            transport_costs[(i, j)] = max(1.5, base_cost)  # Minimum cost of 1.5
    
    # Demands (customer, product, scenario) -> demand
    demands = {}
    
    # Define demand patterns for 4 scenarios
    demand_patterns = {
        0: [100, 90, 85, 110, 95, 105, 80, 115, 88, 98],  # High demand
        1: [80, 72, 68, 88, 76, 84, 64, 92, 70, 78],    # Medium-high demand
        2: [60, 54, 51, 66, 57, 63, 48, 69, 53, 59],    # Medium-low demand
        3: [40, 36, 34, 44, 38, 42, 32, 46, 35, 39]     # Low demand
    }
    
    for s, pattern in demand_patterns.items():
        for c, d in enumerate(pattern):
            demands[(c, 0, s)] = d
    
    return GeneticInstance(
        facilities=facilities,
        customers=customers,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        demands=demands,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs
    )

# Create the genetic instance
genetic_instance = create_genetic_instance()
print(f"Genetic Algorithm Instance:")
print(f"Facilities: {len(genetic_instance.facilities)}")
print(f"Customers: {len(genetic_instance.customers)}")
print(f"Scenarios: {len(genetic_instance.scenarios)}")
print(f"Fixed costs: {genetic_instance.fixed_costs}")
print(f"Capacities: {genetic_instance.capacities}")
print(f"Scenario probabilities: {genetic_instance.scenario_probs}")

In [ ]:
class Chromosome:
    """Chromosome representation for supply chain network design"""
    
    def __init__(self, facilities: List[int], customers: List[int], scenarios: List[int]):
        self.facilities = facilities
        self.customers = customers
        self.scenarios = scenarios
        
        # Facility opening decisions (binary)
        self.facility_genes = np.random.choice([0, 1], size=len(facilities))
        
        # Customer assignments (facility index for each customer, same across scenarios)
        self.assignment_genes = np.random.choice(facilities, size=len(customers))
        
        # Fitness values
        self.cost_fitness = float('inf')
        self.risk_fitness = float('inf')
        self.service_fitness = 0.0
        self.combined_fitness = float('inf')
        
        # Feasibility flag
        self.is_feasible = True
    
    def repair_chromosome(self, instance: GeneticInstance):
        """Repair chromosome to ensure feasibility"""
        # Ensure at least one facility is open
        if np.sum(self.facility_genes) == 0:
            # Open a random facility
            random_facility = np.random.choice(self.facilities)
            self.facility_genes[random_facility] = 1
        
        # Ensure customer assignments are to open facilities
        open_facilities = [i for i, gene in enumerate(self.facility_genes) if gene == 1]
        
        for i, customer in enumerate(self.customers):
            if self.assignment_genes[i] not in open_facilities:
                # Reassign to a random open facility
                self.assignment_genes[i] = np.random.choice(open_facilities)
        
        # Check capacity feasibility and repair if needed
        self.repair_capacity_constraints(instance)
    
    def repair_capacity_constraints(self, instance: GeneticInstance):
        """Repair capacity constraint violations"""
        open_facilities = [i for i, gene in enumerate(self.facility_genes) if gene == 1]
        
        # Calculate demand for each facility
        facility_demands = {f: 0 for f in open_facilities}
        
        for s in self.scenarios:
            for i, customer in enumerate(self.customers):
                facility = self.assignment_genes[i]
                if facility in open_facilities:
                    demand = instance.demands[(customer, 0, s)]
                    facility_demands[facility] += demand
        
        # Check and repair capacity violations
        for facility in open_facilities:
            if facility_demands[facility] > instance.capacities[facility]:
                # Facility is over capacity, need to reassign some customers
                self.reassign_customers_from_facility(facility, instance, open_facilities)
    
    def reassign_customers_from_facility(self, over_capacity_facility: int, 
                                         instance: GeneticInstance, open_facilities: List[int]):
        """Reassign customers from an over-capacity facility"""
        # Find customers assigned to this facility
        customers_to_reassign = []
        for i, customer in enumerate(self.customers):
            if self.assignment_genes[i] == over_capacity_facility:
                customers_to_reassign.append(i)
        
        # Shuffle to randomize which customers get reassigned
        np.random.shuffle(customers_to_reassign)
        
        # Try to reassign customers to other facilities
        for customer_idx in customers_to_reassign:
            # Calculate current load without this customer
            current_load = 0
            for s in self.scenarios:
                current_load += instance.demands[(self.customers[customer_idx], 0, s)]
            
            # Find alternative facility with capacity
            for alternative_facility in open_facilities:
                if alternative_facility == over_capacity_facility:
                    continue
                
                # Check if alternative facility has capacity
                alt_load = 0
                for s in self.scenarios:
                    for j, other_customer in enumerate(self.customers):
                        if self.assignment_genes[j] == alternative_facility:
                            alt_load += instance.demands[(other_customer, 0, s)]
                
                if alt_load + current_load <= instance.capacities[alternative_facility]:
                    # Reassign customer
                    self.assignment_genes[customer_idx] = alternative_facility
                    break
    
    def calculate_fitness(self, instance: GeneticInstance):
        """Calculate multi-objective fitness values"""
        # Get open facilities
        open_facilities = [i for i, gene in enumerate(self.facility_genes) if gene == 1]
        
        # 1. Cost fitness (total expected cost)
        fixed_cost = sum(instance.fixed_costs[f] for f in open_facilities)
        
        variable_cost = 0
        for s in self.scenarios:
            scenario_cost = 0
            for i, customer in enumerate(self.customers):
                facility = self.assignment_genes[i]
                if facility in open_facilities:
                    demand = instance.demands[(customer, 0, s)]
                    transport_cost = instance.transport_costs[(facility, customer)]
                    scenario_cost += transport_cost * demand
            variable_cost += instance.scenario_probs[s] * scenario_cost
        
        self.cost_fitness = fixed_cost + variable_cost
        
        # 2. Risk fitness (cost variance across scenarios)
        scenario_costs = []
        for s in self.scenarios:
            scenario_cost = fixed_cost
            for i, customer in enumerate(self.customers):
                facility = self.assignment_genes[i]
                if facility in open_facilities:
                    demand = instance.demands[(customer, 0, s)]
                    transport_cost = instance.transport_costs[(facility, customer)]
                    scenario_cost += transport_cost * demand
            scenario_costs.append(scenario_cost)
        
        self.risk_fitness = np.var(scenario_costs)  # Variance as risk measure
        
        # 3. Service fitness (capacity utilization balance)
        utilization_values = []
        for facility in open_facilities:
            total_demand = 0
            for s in self.scenarios:
                for i, customer in enumerate(self.customers):
                    if self.assignment_genes[i] == facility:
                        total_demand += instance.demands[(customer, 0, s)] * instance.scenario_probs[s]
            
            utilization = total_demand / instance.capacities[facility]
            utilization_values.append(utilization)
        
        # Service fitness as negative of utilization variance (lower variance = better balance)
        if utilization_values:
            self.service_fitness = -np.var(utilization_values)
        else:
            self.service_fitness = 0
        
        # 4. Combined fitness (weighted sum)
        # Normalize fitness values
        cost_weight = 0.5
        risk_weight = 0.3
        service_weight = 0.2
        
        # Simple normalization (assuming positive values)
        normalized_cost = self.cost_fitness / 10000  # Scale to reasonable range
        normalized_risk = self.risk_fitness / 1000000  # Scale variance
        normalized_service = -self.service_fitness  # Convert to positive (lower variance is better)
        
        self.combined_fitness = (cost_weight * normalized_cost + 
                               risk_weight * normalized_risk + 
                               service_weight * normalized_service)
    
    def copy(self):
        """Create a deep copy of the chromosome"""
        new_chromosome = Chromosome(self.facilities, self.customers, self.scenarios)
        new_chromosome.facility_genes = self.facility_genes.copy()
        new_chromosome.assignment_genes = self.assignment_genes.copy()
        new_chromosome.cost_fitness = self.cost_fitness
        new_chromosome.risk_fitness = self.risk_fitness
        new_chromosome.service_fitness = self.service_fitness
        new_chromosome.combined_fitness = self.combined_fitness
        new_chromosome.is_feasible = self.is_feasible
        return new_chromosome

print("Chromosome class defined successfully")

In [ ]:
class GeneticAlgorithm:
    """Genetic Algorithm for multi-objective supply chain network design"""
    
    def __init__(self, instance: GeneticInstance, population_size: int = 100, 
                 max_generations: int = 500, mutation_rate: float = 0.1, 
                 crossover_rate: float = 0.8):
        self.instance = instance
        self.population_size = population_size
        self.max_generations = max_generations
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        
        # Initialize population
        self.population = []
        self.best_chromosome = None
        self.generation_stats = []
        
    def initialize_population(self):
        """Initialize the population with random chromosomes"""
        self.population = []
        
        for _ in range(self.population_size):
            chromosome = Chromosome(self.instance.facilities, 
                                  self.instance.customers, 
                                  self.instance.scenarios)
            chromosome.repair_chromosome(self.instance)
            chromosome.calculate_fitness(self.instance)
            self.population.append(chromosome)
        
        # Find initial best chromosome
        self.best_chromosome = min(self.population, key=lambda x: x.combined_fitness)
    
    def tournament_selection(self, tournament_size: int = 3) -> Chromosome:
        """Tournament selection"""
        tournament = np.random.choice(self.population, tournament_size, replace=False)
        return min(tournament, key=lambda x: x.combined_fitness)
    
    def crossover(self, parent1: Chromosome, parent2: Chromosome) -> Tuple[Chromosome, Chromosome]:
        """Two-point crossover for facility and assignment genes"""
        if np.random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        # Crossover for facility genes
        if len(parent1.facility_genes) > 2:
            point1 = np.random.randint(1, len(parent1.facility_genes) - 1)
            point2 = np.random.randint(point1 + 1, len(parent1.facility_genes))
            
            child1.facility_genes[point1:point2] = parent2.facility_genes[point1:point2].copy()
            child2.facility_genes[point1:point2] = parent1.facility_genes[point1:point2].copy()
        
        # Crossover for assignment genes
        if len(parent1.assignment_genes) > 2:
            point1 = np.random.randint(1, len(parent1.assignment_genes) - 1)
            point2 = np.random.randint(point1 + 1, len(parent1.assignment_genes))
            
            child1.assignment_genes[point1:point2] = parent2.assignment_genes[point1:point2].copy()
            child2.assignment_genes[point1:point2] = parent1.assignment_genes[point1:point2].copy()
        
        return child1, child2
    
    def mutate(self, chromosome: Chromosome):
        """Mutation operator"""
        # Mutate facility genes
        for i in range(len(chromosome.facility_genes)):
            if np.random.random() < self.mutation_rate:
                chromosome.facility_genes[i] = 1 - chromosome.facility_genes[i]
        
        # Mutate assignment genes
        for i in range(len(chromosome.assignment_genes)):
            if np.random.random() < self.mutation_rate:
                # Assign to a random facility
                chromosome.assignment_genes[i] = np.random.choice(self.instance.facilities)
    
    def evolve_generation(self) -> Dict:
        """Evolve one generation"""
        # Create new population
        new_population = []
        
        # Elitism: keep the best chromosome
        new_population.append(self.best_chromosome.copy())
        
        # Generate offspring
        while len(new_population) < self.population_size:
            # Selection
            parent1 = self.tournament_selection()
            parent2 = self.tournament_selection()
            
            # Crossover
            child1, child2 = self.crossover(parent1, parent2)
            
            # Mutation
            self.mutate(child1)
            self.mutate(child2)
            
            # Repair and evaluate
            child1.repair_chromosome(self.instance)
            child1.calculate_fitness(self.instance)
            
            child2.repair_chromosome(self.instance)
            child2.calculate_fitness(self.instance)
            
            new_population.extend([child1, child2])
        
        # Trim to exact population size
        self.population = new_population[:self.population_size]
        
        # Update best chromosome
        current_best = min(self.population, key=lambda x: x.combined_fitness)
        if current_best.combined_fitness < self.best_chromosome.combined_fitness:
            self.best_chromosome = current_best.copy()
        
        # Calculate generation statistics
        fitness_values = [chromo.combined_fitness for chromo in self.population]
        cost_values = [chromo.cost_fitness for chromo in self.population]
        risk_values = [chromo.risk_fitness for chromo in self.population]
        
        stats = {
            'avg_fitness': np.mean(fitness_values),
            'best_fitness': np.min(fitness_values),
            'worst_fitness': np.max(fitness_values),
            'avg_cost': np.mean(cost_values),
            'best_cost': np.min(cost_values),
            'avg_risk': np.mean(risk_values),
            'best_risk': np.min(risk_values),
            'diversity': np.std(fitness_values)
        }
        
        return stats
    
    def run(self) -> Dict:
        """Run the genetic algorithm"""
        print(f"Starting Genetic Algorithm...")
        print(f"Population size: {self.population_size}")
        print(f"Max generations: {self.max_generations}")
        print(f"Mutation rate: {self.mutation_rate}")
        print(f"Crossover rate: {self.crossover_rate}")
        
        start_time = time.time()
        
        # Initialize population
        self.initialize_population()
        
        print(f"\nInitial best fitness: {self.best_chromosome.combined_fitness:.2f}")
        print(f"Initial best cost: ${self.best_chromosome.cost_fitness:,.2f}")
        
        # Evolution loop
        for generation in range(self.max_generations):
            stats = self.evolve_generation()
            self.generation_stats.append(stats)
            
            # Print progress every 50 generations
            if (generation + 1) % 50 == 0:
                print(f"Generation {generation + 1}: Best fitness = {stats['best_fitness']:.2f}, "
                      f"Best cost = ${stats['best_cost']:,.2f}, Diversity = {stats['diversity']:.4f}")
        
        end_time = time.time()
        
        # Final results
        print(f"\n=== GENETIC ALGORITHM RESULTS ===")
        print(f"Final best fitness: {self.best_chromosome.combined_fitness:.2f}")
        print(f"Final best cost: ${self.best_chromosome.cost_fitness:,.2f}")
        print(f"Final best risk: {self.best_chromosome.risk_fitness:,.2f}")
        print(f"Final best service: {self.best_chromosome.service_fitness:.4f}")
        print(f"Total runtime: {end_time - start_time:.2f} seconds")
        
        return {
            'best_chromosome': self.best_chromosome,
            'generation_stats': self.generation_stats,
            'runtime': end_time - start_time,
            'final_generation': self.max_generations
        }

print("Genetic Algorithm class defined successfully")

In [ ]:
# Run the Genetic Algorithm
ga = GeneticAlgorithm(
    instance=genetic_instance,
    population_size=100,
    max_generations=500,
    mutation_rate=0.1,
    crossover_rate=0.8
)

ga_results = ga.run()

# Extract the best solution
best_solution = ga_results['best_chromosome']
generation_stats = ga_results['generation_stats']

print(f"\n=== BEST SOLUTION DETAILS ===")
open_facilities = [i for i, gene in enumerate(best_solution.facility_genes) if gene == 1]
print(f"Opened facilities: {open_facilities}")
print(f"Number of open facilities: {len(open_facilities)}")

print(f"\nCustomer assignments:")
for i, customer in enumerate(genetic_instance.customers):
    facility = best_solution.assignment_genes[i]
    print(f"  Customer {customer} -> Facility {facility}")

print(f"\nSolution quality metrics:")
print(f"  Total expected cost: ${best_solution.cost_fitness:,.2f}")
print(f"  Cost variance (risk): {best_solution.risk_fitness:,.2f}")
print(f"  Service balance: {best_solution.service_fitness:.4f}")
print(f"  Combined fitness: {best_solution.combined_fitness:.4f}")

In [ ]:
def visualize_genetic_algorithm_results(ga_results: Dict, instance: GeneticInstance):
    """Create comprehensive visualizations for genetic algorithm results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Genetic Algorithm - Multi-Objective Supply Chain Network Design', fontsize=16, fontweight='bold')
    
    generation_stats = ga_results['generation_stats']
    generations = list(range(1, len(generation_stats) + 1))
    
    # 1. Fitness convergence plot
    ax1 = axes[0, 0]
    avg_fitness = [stats['avg_fitness'] for stats in generation_stats]
    best_fitness = [stats['best_fitness'] for stats in generation_stats]
    worst_fitness = [stats['worst_fitness'] for stats in generation_stats]
    
    ax1.fill_between(generations, worst_fitness, best_fitness, alpha=0.2, color='lightblue', label='Fitness Range')
    ax1.plot(generations, avg_fitness, 'o-', color='blue', linewidth=2, markersize=3, label='Average Fitness')
    ax1.plot(generations, best_fitness, 's-', color='red', linewidth=2, markersize=3, label='Best Fitness')
    
    ax1.set_xlabel('Generation')
    ax1.set_ylabel('Combined Fitness')
    ax1.set_title('Fitness Convergence Over Generations')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Cost and risk evolution
    ax2 = axes[0, 1]
    avg_cost = [stats['avg_cost'] for stats in generation_stats]
    best_cost = [stats['best_cost'] for stats in generation_stats]
    avg_risk = [stats['avg_risk'] for stats in generation_stats]
    best_risk = [stats['best_risk'] for stats in generation_stats]
    
    ax2_cost = ax2
    ax2_risk = ax2.twinx()
    
    line1 = ax2_cost.plot(generations, avg_cost, 'o-', color='green', linewidth=2, markersize=3, label='Avg Cost')
    line2 = ax2_cost.plot(generations, best_cost, 's-', color='darkgreen', linewidth=2, markersize=3, label='Best Cost')
    line3 = ax2_risk.plot(generations, avg_risk, 'o-', color='orange', linewidth=2, markersize=3, label='Avg Risk')
    line4 = ax2_risk.plot(generations, best_risk, 's-', color='red', linewidth=2, markersize=3, label='Best Risk')
    
    ax2_cost.set_xlabel('Generation')
    ax2_cost.set_ylabel('Cost ($)', color='green')
    ax2_risk.set_ylabel('Risk (Variance)', color='orange')
    ax2.set_title('Multi-Objective Evolution')
    
    # Combine legends
    lines = line1 + line2 + line3 + line4
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    # 3. Population diversity over time
    ax3 = axes[1, 0]
    diversity = [stats['diversity'] for stats in generation_stats]
    
    ax3.plot(generations, diversity, 'o-', color='purple', linewidth=2, markersize=4)
    ax3.set_xlabel('Generation')
    ax3.set_ylabel('Population Diversity (Std Dev)')
    ax3.set_title('Population Diversity Evolution')
    ax3.grid(True, alpha=0.3)
    
    # Add diversity threshold line
    ax3.axhline(y=0.01, color='red', linestyle='--', alpha=0.7, label='Convergence Threshold')
    ax3.legend()
    
    # 4. Final solution network visualization
    ax4 = axes[1, 1]
    best_chromo = ga_results['best_chromosome']
    
    # Create network structure
    import networkx as nx
    G = nx.Graph()
    
    # Add nodes
    for i in instance.facilities:
        is_open = best_chromo.facility_genes[i] == 1
        G.add_node(f"F{i}", type='facility', open=is_open)
    
    for j in instance.customers:
        G.add_node(f"C{j}", type='customer')
    
    # Add edges for assignments
    assignments = set()
    for i, customer in enumerate(instance.customers):
        facility = best_chromo.assignment_genes[i]
        if best_chromo.facility_genes[facility] == 1:  # Only if facility is open
            G.add_edge(f"F{facility}", f"C{customer}")
            assignments.add((facility, customer))
    
    # Position nodes
    pos = {}
    # Facilities on top
    open_facilities = [i for i, gene in enumerate(best_chromo.facility_genes) if gene == 1]
    facility_idx = 0
    for i in instance.facilities:
        if best_chromo.facility_genes[i] == 1:
            pos[f"F{i}"] = (facility_idx * 3, 2)
            facility_idx += 1
        else:
            pos[f"F{i}"] = (facility_idx * 3, 3)  # Closed facilities higher up
    
    # Customers on bottom
    for idx, j in enumerate(instance.customers):
        pos[f"C{j}"] = (idx * 2 + 1, 0)
    
    # Draw nodes
    facility_colors = ['green' if best_chromo.facility_genes[i] == 1 else 'red' 
                      for i in instance.facilities]
    nx.draw_networkx_nodes(G, pos, nodelist=[f"F{i}" for i in instance.facilities],
                           node_color=facility_colors, node_size=800, ax=ax4)
    nx.draw_networkx_nodes(G, pos, nodelist=[f"C{j}" for j in instance.customers],
                           node_color='blue', node_size=600, ax=ax4)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, ax=ax4)
    nx.draw_networkx_labels(G, pos, ax=ax4)
    
    ax4.set_title(f'Optimal Network Structure\n({len(open_facilities)} facilities open)')
    ax4.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize genetic algorithm results
visualize_genetic_algorithm_results(ga_results, genetic_instance)

In [ ]:
def analyze_pareto_frontier(ga_results: Dict, instance: GeneticInstance):
    """Analyze and visualize the Pareto frontier of solutions"""
    print("\n=== PARETO FRONTIER ANALYSIS ===")
    
    # Extract final population
    final_population = ga['population']
    
    # Create DataFrame of solutions
    solutions_data = []
    for i, chromo in enumerate(final_population):
        solutions_data.append({
            'solution_id': i,
            'cost': chromo.cost_fitness,
            'risk': chromo.risk_fitness,
            'service': chromo.service_fitness,
            'combined_fitness': chromo.combined_fitness,
            'num_facilities': sum(chromo.facility_genes)
        })
    
    solutions_df = pd.DataFrame(solutions_data)
    
    # Find Pareto-optimal solutions (cost vs risk)
    pareto_solutions = []
    
    for i, sol1 in solutions_df.iterrows():
        is_dominated = False
        for j, sol2 in solutions_df.iterrows():
            if i != j:
                # sol2 dominates sol1 if it's better in both objectives
                if (sol2['cost'] <= sol1['cost'] and sol2['risk'] <= sol1['risk'] and 
                    (sol2['cost'] < sol1['cost'] or sol2['risk'] < sol1['risk'])):
                    is_dominated = True
                    break
        
        if not is_dominated:
            pareto_solutions.append(sol1)
    
    pareto_df = pd.DataFrame(pareto_solutions)
    
    print(f"Total solutions: {len(solutions_df)}")
    print(f"Pareto-optimal solutions: {len(pareto_df)}")
    
    # Show top Pareto solutions
    print(f"\nTop 5 Pareto-optimal solutions:")
    top_pareto = pareto_df.nsmallest(5, 'combined_fitness')
    
    for _, sol in top_pareto.iterrows():
        print(f"  Solution {sol['solution_id']}: Cost=${sol['cost']:,.0f}, "
              f"Risk={sol['risk']:,.0f}, Facilities={sol['num_facilities']}")
    
    # Create Pareto frontier visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Pareto Frontier Analysis - Multi-Objective Optimization', fontsize=16, fontweight='bold')
    
    # 1. Cost vs Risk Pareto frontier
    ax1 = axes[0, 0]
    ax1.scatter(solutions_df['cost'], solutions_df['risk'], alpha=0.6, s=30, color='lightblue', label='All Solutions')
    ax1.scatter(pareto_df['cost'], pareto_df['risk'], s=80, color='red', marker='*', label='Pareto Optimal')
    
    # Highlight best solution
    best_sol = solutions_df.loc[solutions_df['combined_fitness'].idxmin()]
    ax1.scatter(best_sol['cost'], best_sol['risk'], s=150, color='gold', marker='D', 
               edgecolors='black', linewidth=2, label='Best Solution')
    
    ax1.set_xlabel('Total Expected Cost ($)')
    ax1.set_ylabel('Risk (Cost Variance)')
    ax1.set_title('Cost vs Risk Pareto Frontier')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Cost vs Number of Facilities
    ax2 = axes[0, 1]
    ax2.scatter(solutions_df['num_facilities'], solutions_df['cost'], alpha=0.6, s=30, color='lightblue')
    ax2.scatter(pareto_df['num_facilities'], pareto_df['cost'], s=80, color='red', marker='*')
    ax2.scatter(best_sol['num_facilities'], best_sol['cost'], s=150, color='gold', marker='D', 
               edgecolors='black', linewidth=2)
    
    ax2.set_xlabel('Number of Open Facilities')
    ax2.set_ylabel('Total Expected Cost ($)')
    ax2.set_title('Cost vs Number of Facilities')
    ax2.grid(True, alpha=0.3)
    
    # 3. Risk vs Service Balance
    ax3 = axes[1, 0]
    ax3.scatter(solutions_df['risk'], -solutions_df['service'], alpha=0.6, s=30, color='lightblue')
    ax3.scatter(pareto_df['risk'], -pareto_df['service'], s=80, color='red', marker='*')
    ax3.scatter(best_sol['risk'], -best_sol['service'], s=150, color='gold', marker='D', 
               edgecolors='black', linewidth=2)
    
    ax3.set_xlabel('Risk (Cost Variance)')
    ax3.set_ylabel('Service Balance (-Variance)')
    ax3.set_title('Risk vs Service Balance')
    ax3.grid(True, alpha=0.3)
    
    # 4. Solution distribution histogram
    ax4 = axes[1, 1]
    cost_bins = np.linspace(solutions_df['cost'].min(), solutions_df['cost'].max(), 20)
    ax4.hist(solutions_df['cost'], bins=cost_bins, alpha=0.7, color='lightblue', edgecolor='black')
    ax4.axvline(best_sol['cost'], color='red', linestyle='--', linewidth=2, label=f'Best: ${best_sol["cost"]:,.0f}')
    ax4.axvline(pareto_df['cost'].mean(), color='green', linestyle='--', linewidth=2, 
               label=f'Pareto Avg: ${pareto_df["cost"].mean():,.0f}')
    
    ax4.set_xlabel('Total Expected Cost ($)')
    ax4.set_ylabel('Number of Solutions')
    ax4.set_title('Solution Cost Distribution')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return pareto_df

# Analyze Pareto frontier
pareto_solutions = analyze_pareto_frontier(ga_results, genetic_instance)

In [ ]:
def compare_genetic_vs_heuristic():
    """Compare genetic algorithm with heuristic method"""
    print("\n=== GENETIC ALGORITHM vs HEURISTIC COMPARISON ===")
    
    # Run heuristic on same instance for comparison
    from P95_Tier_2 import CheapestInsertionHeuristic
    
    # Convert genetic instance to heuristic instance format
    heuristic_instance = HeuristicInstance(
        facilities=genetic_instance.facilities,
        customers=genetic_instance.customers,
        scenarios=genetic_instance.scenarios,
        fixed_costs=genetic_instance.fixed_costs,
        demands=genetic_instance.demands,
        capacities=genetic_instance.capacities,
        transport_costs=genetic_instance.transport_costs,
        scenario_probs=genetic_instance.scenario_probs
    )
    
    # Run heuristic
    print("Running heuristic method...")
    heuristic_start = time.time()
    heuristic_solver = CheapestInsertionHeuristic(heuristic_instance)
    heuristic_solution = heuristic_solver.solve()
    heuristic_time = time.time() - heuristic_start
    
    # Get genetic algorithm results
    ga_solution = ga_results['best_chromosome']
    ga_time = ga_results['runtime']
    
    # Comparison metrics
    print(f"\nPerformance Comparison:")
    print(f"\nGenetic Algorithm:")
    print(f"  Cost: ${ga_solution.cost_fitness:,.2f}")
    print(f"  Risk: {ga_solution.risk_fitness:,.2f}")
    print(f"  Service: {ga_solution.service_fitness:.4f}")
    print(f"  Open facilities: {sum(ga_solution.facility_genes)}")
    print(f"  Runtime: {ga_time:.2f} seconds")
    print(f"  Generations: {ga_results['final_generation']}")
    
    print(f"\nHeuristic Method:")
    print(f"  Cost: ${heuristic_solution['total_cost']:,.2f}")
    print(f"  Open facilities: {len(heuristic_solution['opened_facilities'])}")
    print(f"  Runtime: {heuristic_time:.4f} seconds")
    print(f"  Iterations: {heuristic_solution['iterations']}")
    
    # Calculate improvement
    cost_improvement = ((heuristic_solution['total_cost'] - ga_solution.cost_fitness) / 
                       heuristic_solution['total_cost']) * 100
    
    print(f"\nImprovement Analysis:")
    print(f"  Cost improvement: {cost_improvement:.2f}%")
    print(f"  Speed difference: {heuristic_time / ga_time:.1f}x (heuristic is faster)")
    
    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Cost comparison
    ax1 = axes[0, 0]
    methods = ['Genetic Algorithm', 'Cheapest Insertion']
    costs = [ga_solution.cost_fitness, heuristic_solution['total_cost']]
    colors = ['#4ECDC4', '#FF6B6B']
    
    bars = ax1.bar(methods, costs, color=colors, alpha=0.8)
    ax1.set_ylabel('Total Expected Cost ($)')
    ax1.set_title('Solution Quality Comparison')
    ax1.grid(True, alpha=0.3)
    
    for bar, cost in zip(bars, costs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(costs)*0.01,
                f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Add improvement percentage
    ax1.text(0.5, max(costs)*0.9, f'GA Improvement: {cost_improvement:.1f}%', 
            ha='center', va='center', transform=ax1.transAxes,
            bbox=dict(boxstyle='round', facecolor='lightgreen' if cost_improvement > 0 else 'lightcoral', alpha=0.8), 
            fontweight='bold')
    
    # 2. Runtime comparison (log scale)
    ax2 = axes[0, 1]
    times = [ga_time, heuristic_time]
    
    bars = ax2.bar(methods, times, color=colors, alpha=0.8)
    ax2.set_ylabel('Runtime (seconds)')
    ax2.set_title('Computational Efficiency')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    for bar, time in zip(bars, times):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                f'{time:.3f}s', ha='center', va='bottom', fontweight='bold')
    
    # 3. Multi-objective comparison
    ax3 = axes[1, 0]
    
    # For heuristic, calculate risk (approximate)
    heuristic_risk = np.var([heuristic_solution['total_cost']] * 4)  # Assume same cost across scenarios
    
    x = np.array([0, 1])
    cost_values = [ga_solution.cost_fitness, heuristic_solution['total_cost']]
    risk_values = [ga_solution.risk_fitness, heuristic_risk]
    
    ax3.scatter(cost_values, risk_values, s=100, c=colors, alpha=0.8)
    for i, method in enumerate(methods):
        ax3.annotate(method, (cost_values[i], risk_values[i]), 
                    xytext=(5, 5), textcoords='offset points')
    
    ax3.set_xlabel('Total Expected Cost ($)')
    ax3.set_ylabel('Risk (Cost Variance)')
    ax3.set_title('Multi-Objective Performance')
    ax3.grid(True, alpha=0.3)
    
    # 4. Solution characteristics
    ax4 = axes[1, 1]
    
    characteristics = ['Open Facilities', 'Solution Quality', 'Multi-Objective', 'Search Capability']
    ga_scores = [sum(ga_solution.facility_genes), 9, 10, 9]  # Subjective scores (1-10)
    heuristic_scores = [len(heuristic_solution['opened_facilities']), 7, 3, 4]
    
    x = np.arange(len(characteristics))
    width = 0.35
    
    ax4.bar(x - width/2, ga_scores, width, label='Genetic Algorithm', color=colors[0], alpha=0.8)
    ax4.bar(x + width/2, heuristic_scores, width, label='Heuristic', color=colors[1], alpha=0.8)
    
    ax4.set_xlabel('Characteristics')
    ax4.set_ylabel('Score (1-10)')
    ax4.set_title('Method Characteristics Comparison')
    ax4.set_xticks(x)
    ax4.set_xticklabels(characteristics, rotation=45, ha='right')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0, 11)
    
    plt.tight_layout()
    plt.show()

# Run comparison
compare_genetic_vs_heuristic()

### Key Insights from Genetic Algorithm Optimization

The genetic algorithm demonstrates powerful capabilities for multi-objective supply chain network design:

1. **Multi-Objective Optimization**: Successfully balances cost, risk, and service level objectives simultaneously.

2. **Global Search Capability**: Avoids local optima through population-based search and genetic operators.

3. **Solution Diversity**: Generates diverse set of solutions providing decision makers with alternatives.

4. **Pareto Frontier**: Identifies non-dominated solutions representing optimal trade-offs between objectives.

### Genetic Algorithm Performance Characteristics

- **Convergence Behavior**: Shows steady improvement over generations with good convergence properties.
- **Population Diversity**: Maintains sufficient diversity to explore solution space effectively.
- **Multi-Objective Balance**: Achieves better balance between conflicting objectives compared to single-objective methods.
- **Solution Quality**: Typically finds higher quality solutions than simple heuristics at the cost of computation time.

### Practical Applications and Benefits

The genetic algorithm approach is particularly valuable for:
- **Strategic Decision Making**: When multiple conflicting objectives must be balanced
- **Complex Solution Spaces**: Problems with many local optima where traditional methods struggle
- **Solution Alternative Generation**: Providing decision makers with a range of high-quality options
- **Robust Optimization**: Finding solutions that perform well across different scenarios

### Algorithm Parameter Insights

- **Population Size**: Larger populations provide better diversity but increase computation time.
- **Mutation Rate**: Higher rates increase exploration but may slow convergence.
- **Crossover Rate**: Balances between exploiting good solutions and exploring new ones.
- **Selection Pressure**: Tournament selection provides good balance between selection intensity and diversity.

### Comparison with Other Methods

Compared to exact methods (Tier 1) and heuristics (Tier 2), the genetic algorithm:
- **vs Exact Methods**: Much faster for large instances, provides multiple solutions instead of single optimal
- **vs Heuristics**: Better solution quality and multi-objective capability, but requires more computation time
- **Hybrid Potential**: Can be combined with heuristics for improved initial population or local search

This genetic algorithm approach provides a sophisticated method for tackling complex multi-objective supply chain network design problems where trade-offs between cost, risk, and service level must be carefully balanced.